# 02 — Feature Engineering

Starts from the cleaned dataset saved in notebook 01. No cleaning is repeated here.

Adds the columns the analysis needs: revenue per line, and date parts for
trend and seasonality charts.

In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/retail_clean.parquet")
df.shape

(1021128, 9)

In [2]:
df.dtypes

Invoice                    str
StockCode                  str
Description                str
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID              Int64
Country                    str
is_cancelled              bool
dtype: object

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancelled
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False


## Revenue

`Revenue = Quantity × Price` — the single most important derived column. Every
revenue figure in the EDA, SQL and dashboard traces back to this line.

Cancelled invoices carry negative quantities, so their revenue is naturally
negative. That is intentional: it makes lost revenue measurable instead of hidden.

In [4]:
df["Revenue"] = df["Quantity"] * df["Price"]
df[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [5]:
df.groupby("is_cancelled")["Revenue"].sum().round(2)

is_cancelled
False    19642692.15
True      -716425.97
Name: Revenue, dtype: float64

## Date parts

`InvoiceDate` is a single timestamp. Charts need it broken into the pieces
people actually ask questions about: which month, which weekday, which hour.

`year_month` is stored as text ("2009-12") rather than a pandas Period so it
survives the trip to parquet, PostgreSQL and Power BI unchanged. The
zero-padded format still sorts correctly.

In [6]:
df["year"] = df["InvoiceDate"].dt.year
df["month"] = df["InvoiceDate"].dt.month
df["year_month"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["day_of_week"] = df["InvoiceDate"].dt.day_name()
df["hour"] = df["InvoiceDate"].dt.hour

In [7]:
df[["InvoiceDate", "year", "month", "year_month", "day_of_week", "hour"]].head()

,InvoiceDate,year,month,year_month,day_of_week,hour
0,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7
1,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7
2,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7
3,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7
4,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7


In [8]:
df["year_month"].nunique(), df["year_month"].min(), df["year_month"].max()

(25, '2009-12', '2011-12')

In [9]:
df["day_of_week"].value_counts()

day_of_week
Thursday     193919
Tuesday      189354
Monday       181463
Wednesday    175412
Friday       147903
Sunday       132677
Saturday        400
Name: count, dtype: int64